<a href="https://colab.research.google.com/github/JediricX/desercionIA/blob/main/Alerta_temprana_abandono_universitario_1_EDA_OULAD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Modelo IA para la detección temprana del abandono universitario basado en dataset OULAD

Objetivo del modelo:

Diseñar e implementar un Sistema de Alerta Temprana basado en Inteligencia Artificial que permita predecir el riesgo de abandono en programas universitarios a distancia, mediante la comparación de algoritmos de clasificación supervisada (Random Forest, Árboles de Decisión, XGBoost y Regresión Logística), con el fin de ofrecer a las instituciones educativas una herramienta preventiva para la retención estudiantil.

Autores:


*   Richard Humberto Campos Ballesteros
*   José Andía



# **1. Importación de librerías**

In [ ]:
# Manejo de datos
import pandas as pd
import numpy as np
import os

# Preprocesamiento
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Métricas
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.metrics import RocCurveDisplay

# Desbalance de clases
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


# **2. Carga del dataset OULAD**

In [ ]:
# Install dependencies as needed:
#!pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

#Assessments
assessments = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anlgrbz/student-demographics-online-education-dataoulad",
    "assessments.csv"
)

#Courses
courses = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anlgrbz/student-demographics-online-education-dataoulad",
    "courses.csv"
)

#student_assessment
student_assessments = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anlgrbz/student-demographics-online-education-dataoulad",
    "studentAssessment.csv"
)

#student_info
student_info = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anlgrbz/student-demographics-online-education-dataoulad",
    "studentInfo.csv"
)

#student_registration
student_registration = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anlgrbz/student-demographics-online-education-dataoulad",
    "studentRegistration.csv"
)

#vle
vle = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anlgrbz/student-demographics-online-education-dataoulad",
    "vle.csv"
)

#student_vle
student_vle = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anlgrbz/student-demographics-online-education-dataoulad",
    "studentVle.csv"
)


# **3. Fase de análisis de datos (EDA)**

In [ ]:
print("### 3.1. Descripción general de los datasets")

# Función para mostrar información básica de cada DataFrame
def display_df_info(df, name):
    print(f"\n--- {name} DataFrame ---")
    print(f"Shape: {df.shape}")
    print(f"Head:\n{df.head()}")
    print(f"\nInfo:")
    df.info()
    # Contando el numero de valores perdidos en cada columna
    print("Número de valores nulos por cada columna")
    print(f"del dataset {name}")
    print("-------"*10)
    nan_count = df.isna().sum()
    # Print the result
    print(nan_count)
    print("-------"*10)
    if df.select_dtypes(include=np.number).columns.any():
        print(f"\nDescription of Numerical Columns:\n{df.describe()}")


display_df_info(assessments, "Assessments")
display_df_info(courses, "Courses")
display_df_info(student_assessments, "Student Assessments")
display_df_info(student_info, "Student Info")
display_df_info(student_registration, "Student Registration")
display_df_info(vle, "VLE")
display_df_info(student_vle, "Student VLE")

print("\n### 3.2. Análisis de la variable objetivo (`final_result`)")
# Se recomienda crear variable 'dropout' y eliminar final_result en el dataset
print("Distribución de 'final_result':")
print(student_info['final_result'].value_counts())
print("\nPorcentaje de distribución de 'final_result':")
print(student_info['final_result'].value_counts(normalize=True) * 100)

# Librerías para la visualización de la variable objetivo
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.countplot(data=student_info, x='final_result', order=student_info['final_result'].value_counts().index)
plt.title('Distribución de Final Result')
plt.xlabel('Final Result')
plt.ylabel('Número de estudiantes')
plt.show()

print("\nBasado en el objetivo ('abandono'), 'Withdrawn' es la salida clave para la predicción.")

print("\n### 3.3. Exploración de variable clave y uniones iniciales para predicción temprana")

print("#### 3.3.1. Fusionando información y registro de estudiantes")
# Se fusionan student_info y student_registration para obtener una vista combinada
# Esta unión es crucial ya que 'final_result' se encuentra en el dataset student_info y 'date_registration' está en el dataset student_registration
student_data = pd.merge(student_info, student_registration,
                        on=['id_student', 'code_module', 'code_presentation'],
                        how='left')

print("Encabezado del dataset fusionado 'student_data' (student_info + student_registration):\n")
print(student_data.head())
print("\nInformación del dataset 'student_data':")
student_data.info()
print("\nValores perdidos en 'student_data':")
print(student_data.isnull().sum()[student_data.isnull().sum() > 0])

print("\n#### 3.3.2. Analizando características demográficas y de antecedentes vs. Final Result")
demographic_cols = ['gender', 'region', 'highest_education', 'age_band', 'num_of_prev_attempts', 'disability']

for col in demographic_cols:
    plt.figure(figsize=(10, 5))
    sns.countplot(data=student_data, x=col, hue='final_result', palette='viridis')
    plt.title(f'Final Result by {col}')
    plt.xlabel(col)
    plt.ylabel('Number of Students')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Final Result')
    plt.tight_layout()
    plt.show()

# Análisis 'studied_credits'
plt.figure(figsize=(10, 6))
sns.boxplot(data=student_data, x='final_result', y='studied_credits', palette='viridis')
plt.title('Studied Credits by Final Result')
plt.xlabel('Final Result')
plt.ylabel('Studied Credits')
plt.show()

print("\n#### 3.3.3. Analizando datos de registro (`date_registration`)")
# 'date_registration' es el número de días antes del inicio de la presentación del módulo.
# Valores negativos significan registro antes del comienzo del módulo, positivos después del inicio.
plt.figure(figsize=(10, 6))
sns.histplot(student_data['date_registration'].dropna(), kde=True, bins=50)
plt.title('Distribución de Fecha de registro (Días antes/despues del inicio del módulo)')
plt.xlabel('Días desde el inicio del módulo (Negativo = Antes, Positivo = Después)')
plt.ylabel('Número de estudiantes')
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(data=student_data, x='final_result', y='date_registration', palette='viridis')
plt.title('Fecha de registro por Final Result')
plt.xlabel('Final Result')
plt.ylabel('Días desde inicio del módulo')
plt.show()

print("\n#### 3.3.4. Explorando el desempeño de la evaluación temprana")
# Se fusiona el dataset student_assessments con assessments para obtener detalles de evaluación
student_assessments_details = pd.merge(student_assessments, assessments,
                                       on='id_assessment', how='left')

# Se fusiona con student_data para enlazar con la variable objetivo final_result (más adelante dropout) y datos demográficos
full_assessment_data = pd.merge(student_data, student_assessments_details,
                                on=['id_student', 'code_module', 'code_presentation'],
                                how='left')

print("Encabezado de 'full_assessment_data':")
print(full_assessment_data.head())
full_assessment_data.info()

# Enfoque en evaluaciones tempranas: Se filtra por evaluaciones realizadas temprano en el módulo(date < 28 días)
early_assessments = full_assessment_data[full_assessment_data['date'] <= 28]

# Se calcula puntuacón promedio para evaulaciones tempranas para cada estudiante
early_avg_scores = early_assessments.groupby(['id_student', 'code_module', 'code_presentation'])['score'].mean().reset_index()
early_avg_scores.rename(columns={'score': 'early_avg_score'}, inplace=True)

# Se fusiona early_avg_scores con el student_data
student_data = pd.merge(student_data, early_avg_scores,
                        on=['id_student', 'code_module', 'code_presentation'],
                        how='left')

plt.figure(figsize=(10, 6))
sns.boxplot(data=student_data, x='final_result', y='early_avg_score', palette='viridis')
plt.title('Puntuación promedio de evaluación inicial según Final Result')
plt.xlabel('Final Result')
plt.ylabel('Promedio de puntuación temprana')
plt.show()

print("\n#### 3.3.5. Explorando Actividad VLE temprana")
# Se fusiona los datasets student_vle y vle para obtener tipos de actividad VLE
student_vle_details = pd.merge(student_vle, vle,
                               on=['id_site', 'code_module', 'code_presentation'],
                               how='left')

# Se fusiona student_data con student_vle_details para conenctar la variable objetivo final_result (dropout)
full_vle_data = pd.merge(student_data, student_vle_details,
                         on=['id_student', 'code_module', 'code_presentation'],
                         how='left')

print("Encabezado de 'full_vle_data':")
print(full_vle_data.head())
full_vle_data.info()

# Enfoque en actividad VLE tremprana: Se filtra por actividad que ocurre temprano en el módulo (date < 28 días)
early_vle_activity = full_vle_data[full_vle_data['date'] <= 28] # date pertenece al dataset student_vle

# Se calcula clicks totales para actividad VLE temprana para cada estudiante
early_total_clicks = early_vle_activity.groupby(['id_student', 'code_module', 'code_presentation'])['sum_click'].sum().reset_index()
early_total_clicks.rename(columns={'sum_click': 'early_total_clicks'}, inplace=True)

# Se fusiona early_total_clicks con student_data
student_data = pd.merge(student_data, early_total_clicks,
                        on=['id_student', 'code_module', 'code_presentation'],
                        how='left')

plt.figure(figsize=(10, 6))
sns.boxplot(data=student_data, x='final_result', y='early_total_clicks', palette='viridis')
plt.title('Clics totales tempranos de VLE por final_result')
plt.xlabel('Final Result')
plt.ylabel('Clicks tempranos totales')
plt.ylim(0, student_data['early_total_clicks'].quantile(0.99)) # Se recortan outliers para mejor visualización
plt.show()

print("\n--- EDA Summary and Key Variables for Early Prediction ---")
print("Variables identified as potentially important for early prediction include:")
print("- **Demographic and Background:** gender, region, highest_education, age_band, num_of_prev_attempts, disability, studied_credits. These are available at the start.")
print("- **Registration Behavior:** `date_registration` (how early/late a student registers). This is an early indicator.")
print("- **Early Assessment Performance:** Average scores from assessments taken in the first few weeks/months. This requires monitoring initial academic engagement.")
print("- **Early VLE Engagement:** Total clicks or activity types within the Virtual Learning Environment during the initial period of the course. This also requires monitoring early activity.")
print("\nFurther steps would involve more detailed feature engineering from these raw variables, handling missing values, and preparing the data for model training.")

**3.1 Distribución variable objetivo**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Crear una nueva variable para agrupar los resultados
student_info['grouped_final_result'] = student_info['final_result'].apply(lambda x: 'Withdrawn' if x == 'Withdrawn' else 'Not Withdrawn')

# Calcular la distribución de la nueva variable
distribution = student_info['grouped_final_result'].value_counts()

# Generar el gráfico de pastel
plt.figure(figsize=(8, 8))
plt.pie(distribution, labels=distribution.index, autopct='%1.1f%%', startangle=90, colors=['#ff9999','#66b3ff'])
plt.title('Distribución de la Variable Objetivo: Abandono vs. No Abandono')
plt.axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

# Opcional: Eliminar la columna temporal si no se necesita más
student_info.drop(columns=['grouped_final_result'], inplace=True)